# Filter Full Hair Images

Create a stronger CelebA subset that keeps only images with full hairstyle visibility.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from app.ml.celeba_hair_rich import enrich_full_hair_records, full_hair_summary, read_jsonl, write_jsonl

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
SOURCE_DATASET_DIR = BACKEND_ROOT / 'data' / 'datasets' / 'celeba_hair_rich'
OUTPUT_DATASET_DIR = BACKEND_ROOT / 'data' / 'datasets' / 'celeba_full_hair'

SOURCE_MANIFESTS = {
    'train': SOURCE_DATASET_DIR / 'train_confidence_filtered.jsonl',
    'val': SOURCE_DATASET_DIR / 'val_confidence_filtered.jsonl',
    'test': SOURCE_DATASET_DIR / 'test_confidence_filtered.jsonl',
}

OUTPUT_ENRICHED = {
    split: OUTPUT_DATASET_DIR / f'{split}_enriched.jsonl'
    for split in SOURCE_MANIFESTS
}
OUTPUT_FILTERED = {
    split: OUTPUT_DATASET_DIR / f'{split}.jsonl'
    for split in SOURCE_MANIFESTS
}
SUMMARY_PATH = OUTPUT_DATASET_DIR / 'summary.json'

IMAGE_WIDTH = 178
IMAGE_HEIGHT = 218
MIN_FULL_HAIR_SCORE = 0.82
ENFORCE_GENDER_BALANCE_REPORT = True

SOURCE_MANIFESTS

{'train': WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/data/datasets/celeba_hair_rich/train_confidence_filtered.jsonl'),
 'val': WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/data/datasets/celeba_hair_rich/val_confidence_filtered.jsonl'),
 'test': WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/data/datasets/celeba_hair_rich/test_confidence_filtered.jsonl')}

In [3]:
source_records = {
    split: read_jsonl(path)
    for split, path in SOURCE_MANIFESTS.items()
}

split_summaries = {}
enriched_records = {}
filtered_records = {}

for split, records in source_records.items():
    enriched = enrich_full_hair_records(records, image_width=IMAGE_WIDTH, image_height=IMAGE_HEIGHT)
    accepted = [
        row for row in enriched
        if row.get('full_hair_candidate') and float(row.get('full_hair_score', 0.0)) >= MIN_FULL_HAIR_SCORE
    ]

    enriched_records[split] = enriched
    filtered_records[split] = accepted

    split_summaries[split] = {
        **full_hair_summary(enriched),
        'accepted_after_score_gate': len(accepted),
        'accepted_after_score_ratio': round(len(accepted) / max(len(enriched), 1), 4),
    }

pd.DataFrame(split_summaries).T

,total_records,full_hair_candidates,accepted_ratio,avg_full_hair_score,flag_counts,accepted_after_score_gate,accepted_after_score_ratio
train,1655,148,0.0894,0.7713,"{'hair_too_low': 958, 'touches_bottom_edge': 7...",148,0.0894
val,373,30,0.0804,0.7711,"{'hair_too_low': 207, 'touches_bottom_edge': 1...",30,0.0804
test,413,31,0.0751,0.7691,"{'hair_too_low': 237, 'touches_right_edge': 20...",31,0.0751


In [4]:
OUTPUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)

for split, rows in enriched_records.items():
    write_jsonl(rows, OUTPUT_ENRICHED[split])

for split, rows in filtered_records.items():
    write_jsonl(rows, OUTPUT_FILTERED[split])

summary_payload = {
    'source_dataset': 'celeba_hair_rich_confidence_filtered',
    'output_dataset': 'celeba_full_hair',
    'image_size': {'width': IMAGE_WIDTH, 'height': IMAGE_HEIGHT},
    'min_full_hair_score': MIN_FULL_HAIR_SCORE,
    'splits': split_summaries,
    'output_manifests': {split: str(path) for split, path in OUTPUT_FILTERED.items()},
    'output_enriched_manifests': {split: str(path) for split, path in OUTPUT_ENRICHED.items()},
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2), encoding='utf-8')

pd.Series({
    'summary_path': str(SUMMARY_PATH),
    'train_records': len(filtered_records['train']),
    'val_records': len(filtered_records['val']),
    'test_records': len(filtered_records['test']),
})

summary_path     D:\Projects\Personal Projects\Hairstyle Recomm...
train_records                                                  148
val_records                                                     30
test_records                                                    31
dtype: object

In [5]:
preview_rows = []
for split, rows in filtered_records.items():
    preview_rows.extend(rows[:2])

preview_columns = [
    'image_id',
    'partition',
    'gender_label',
    'full_hair_score',
    'quality_score',
]
display(pd.DataFrame(preview_rows)[preview_columns])

if ENFORCE_GENDER_BALANCE_REPORT:
    gender_frames = []
    for split, rows in filtered_records.items():
        frame = pd.DataFrame(rows)
        if frame.empty:
            continue
        counts = frame['gender_label'].value_counts().rename_axis('gender_label').reset_index(name='count')
        counts.insert(0, 'split', split)
        gender_frames.append(counts)
    if gender_frames:
        display(pd.concat(gender_frames, ignore_index=True))


,image_id,partition,gender_label,full_hair_score,quality_score
0,000011.jpg,train,female,1.00,1.0
1,000023.jpg,train,male,0.85,1.0
2,162791.jpg,val,male,1.00,1.0
3,162793.jpg,val,male,1.00,1.0
4,182657.jpg,test,female,1.00,1.0
5,182673.jpg,test,male,1.00,1.0


,split,gender_label,count
0,train,male,82
1,train,female,66
2,val,female,16
3,val,male,14
4,test,female,17
5,test,male,14
